# change mp4 audio from stereo to mono and NORM


In [2]:
# -----######-----###### MONO & NORMALIZE MP4 AUDIO IN FOLDER -----######-----###### #
import subprocess
from pathlib import Path
from tqdm import tqdm

def _video_3107_vmono_norm_GET_cleanfolder(folder_path):
    folder = Path(folder_path)
    if not folder.exists():
        print("❌ Folder not found.")
        return
    
    video_paths = [p for p in folder.glob("*.mp4") if not p.name.startswith('._')]
    
    for vid_path in tqdm(video_paths, desc="🎬 Processing Videos"):
        temp_mono = vid_path.with_name("temp_audio_mono.wav")
        temp_norm = vid_path.with_name("temp_audio_norm.wav")
        temp_final = vid_path.with_name(f"temp_{vid_path.name}")

        # 1. Extract mono audio
        cmd1 = [
            "ffmpeg", "-i", str(vid_path),
            "-vn", "-ac", "1", "-ar", "44100", "-y",
            str(temp_mono)
        ]
        
        # 2. Normalize audio (peak normalization only)
        cmd2 = [
            "ffmpeg-normalize", str(temp_mono),
            "-o", str(temp_norm),
            "-nt", "peak",  # peak normalization
            "-f",           # force overwrite
            "-c:a", "pcm_s16le"
        ]

        # 3. Rebuild video with normalized mono audio
        cmd3 = [
            "ffmpeg", "-i", str(vid_path), "-i", str(temp_norm),
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "copy", "-c:a", "aac",
            "-y", str(temp_final)
        ]

        try:
            subprocess.run(cmd1, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            subprocess.run(cmd2, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            subprocess.run(cmd3, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

            # Replace original
            vid_path.unlink()
            temp_final.rename(vid_path)

        except subprocess.CalledProcessError as e:
            print(f"⚠️ Error with {vid_path.name}: {e}")
        finally:
            # Clean temp
            for f in [temp_mono, temp_norm]:
                if f.exists():
                    f.unlink()



In [ ]:
# Example folder path
folder_path = "/Users/yerik/Desktop/input_vids"

_video_3107_vmono_norm_GET_cleanfolder(folder_path)
